# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjds22/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The March 2026 dataset contains 3,611,061 observations with GSC data available. Impressions are heavily right-skewed: the median is 16, compared with a mean of 77.72 and a maximum of 40,084. Clicks are sparse, with a median of zero and a mean of 0.23. Average position is also widely distributed, with a median of 7.50 and a 95th percentile of 62.75. These distributions suggest that visibility varies substantially, so averages alone may not represent a typical observation.

In [5]:
import duckdb
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

audit = con.sql(f"""
SELECT
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {fact_daily}
WHERE month = '2026-03'
  AND gsc_data_available = TRUE
""").df()

print("Rows analysed:", len(audit))
display(audit[
    ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows analysed: 3611061


,gsc_impressions,gsc_clicks,gsc_avg_position
count,3611061.00,3611061.00,3611061.00
mean,77.72,0.23,15.83
std,249.87,1.28,19.86
min,1.00,0.00,0.00
25%,4.00,0.00,3.74
50%,16.00,0.00,7.50
75%,62.00,0.00,20.20
90%,185.00,1.00,43.00
95%,335.00,1.00,62.75
max,40084.00,274.00,498.00


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal 1 — Low impressions: The proportion of observations with fewer than 100 impressions measures how common low visibility is. A high proportion supports using impressions as a prioritisation signal, but does not prove that content needs improvement.

Signal 2 — Poor average position: The proportion of observations with an average position of 20 or worse measures how common weak search positions are. This is a visibility indicator, not proof of the cause.

Signal 3 — Position and impressions:** Comparing impressions across position groups tests whether weaker average positions are associated with lower visibility. This is an association test, not evidence of causation. bold text

In [6]:

import numpy as np
import pandas as pd

n = len(audit)

print("SIGNAL 1 — LOW IMPRESSIONS")
print("Under 100 impressions:", round((audit["gsc_impressions"] < 100).mean() * 100, 2), "%")

print("\nSIGNAL 2 — POOR POSITION")
print("Average position >= 20:", round((audit["gsc_avg_position"] >= 20).mean() * 100, 2), "%")

print("\nSIGNAL 3 — POSITION VS IMPRESSIONS")
audit["position_group"] = pd.cut(
    audit["gsc_avg_position"],
    bins=[-np.inf, 10, 20, np.inf],
    labels=["Position < 10", "Position 10–20", "Position > 20"],
    right=False
)
print(audit.groupby("position_group", observed=False).agg(
    rows=("gsc_impressions", "size"),
    median_impressions=("gsc_impressions", "median"),
    mean_impressions=("gsc_impressions", "mean"),
    mean_clicks=("gsc_clicks", "mean")
).round(2))


SIGNAL 1 — LOW IMPRESSIONS
Under 100 impressions: 82.32 %

SIGNAL 2 — POOR POSITION
Average position >= 20: 25.27 %

SIGNAL 3 — POSITION VS IMPRESSIONS
                   rows  median_impressions  mean_impressions  mean_clicks
position_group                                                            
Position < 10   2154941                20.0             88.98         0.30
Position 10–20   543782                18.0             54.16         0.17
Position > 20    912338                 9.0             65.16         0.09


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [7]:

print(con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df().to_string(index=False))


Empty DataFrame
Columns: [table_name]
Index: []


Verdict: NOT TESTABLE DIRECTLY. The warehouse release contains observable search-performance measurements, but FlyRank's internal product decision flags are deliberately excluded. Therefore, this analysis cannot directly verify whether an existing product flag's assumptions hold. Instead, it tests observable proxies, including low impressions and poor average position, and evaluates whether these provide useful evidence for prioritising content review. These proxies are not equivalent to the actual product flags.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The analysis measures how frequently low impressions and weak search positions occur in the observed data. These signals can help prioritise content for human review, but they do not establish why a page performs poorly or guarantee that changing it will improve performance. Recommendations should be treated as decision-support candidates and checked against the relevant content context.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.